# SILVER-CUSTOMERS

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

In [0]:
df = spark.table("bronze.customers").select("*")

In [0]:
df.limit(5).display()

In [0]:
df.count()

In [0]:
df.select(countDistinct("customer_id").alias("unique_count")).display()

In [0]:
df.select(countDistinct("customer_unique_id").alias("unique_count")).display()

In [0]:
df.limit(5).display()

### Step 1: Dedup

In [0]:
w = Window.partitionBy("customer_id").orderBy(col("_load_timestamp").desc())

df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

In [0]:
df.count()

### Step 2: Casting

In [0]:
df = df.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("string"))
df.printSchema()

### Step 3: NULL check

In [0]:
df = df.filter(col("customer_id").isNotNull())
df.count()

### Step 4: write to Silver table

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
df.write.format("delta").mode("overwrite").option("inferSchema", True).saveAsTable("silver.customers")
spark.table("silver.customers").count()